# 02 — LoRA, IP-Adapter & DreamBooth Exploration

This notebook explores three of the most powerful **consistency techniques** for Stable Diffusion:

| Technique | Training required? | Use case |
|---|---|---|
| **IP-Adapter** | ❌ No | Consistent subject/style from a reference image |
| **LoRA** | ✅ Yes (lightweight) | Specific character or style, fast to train |
| **DreamBooth** | ✅ Yes (heavier) | High-fidelity subject binding |

**References:**
- IP-Adapter: https://arxiv.org/abs/2308.06721
- LoRA: https://arxiv.org/abs/2106.09685
- DreamBooth: https://arxiv.org/abs/2208.12242
- Diffusers adapters guide: https://huggingface.co/docs/diffusers/using-diffusers/loading_adapters

## 0. Setup

In [ ]:
import sys

sys.path.insert(0, '..')

from pathlib import Path

from IPython.display import display
from PIL import Image

OUTPUT_DIR = Path('../assets/outputs/02_adapters')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL_ID = 'runwayml/stable-diffusion-v1-5'

---
## Part A — IP-Adapter

IP-Adapter injects a CLIP image embedding into the UNet cross-attention layers.
No training is needed — just provide a reference image.

In [ ]:
from src.image.ip_adapter import IPAdapterPipeline

# TODO: Fill in src/image/ip_adapter.py before running.
ip_pipe = IPAdapterPipeline(
    base_model_id=BASE_MODEL_ID,
    ip_adapter_model_id='h94/IP-Adapter',
)
print('IP-Adapter pipeline loaded.')

In [ ]:
# Replace with your own reference image path
REFERENCE_PATH = '../assets/reference/my_character.png'

try:
    reference_image = Image.open(REFERENCE_PATH).convert('RGB')
    display(reference_image)
except FileNotFoundError:
    print(f'Reference image not found at {REFERENCE_PATH}.')
    print('Place a reference image there and re-run this cell.')
    reference_image = None

In [ ]:
# IP-Adapter scale sweep — how much the reference image influences the output
if reference_image is not None:
    for scale in [0.2, 0.5, 0.8, 1.0]:
        img = ip_pipe.generate(
            prompt='a character exploring a magical forest, cinematic lighting',
            reference_image=reference_image,
            ip_adapter_scale=scale,
            seed=42,
            num_inference_steps=30,
        )
        img.save(OUTPUT_DIR / f'ip_adapter_scale_{scale:.1f}.png')
        print(f'ip_adapter_scale={scale}')
        display(img)

---
## Part B — LoRA

LoRA adapters are fine-tuned weight deltas.  Load a pre-trained LoRA from the Hub
or train your own using `src/training/train_lora.py`.

In [ ]:
from src.image.lora import LoRAPipeline

# TODO: Replace with a real LoRA path or Hub ID.
# Example Hub LoRA: 'CiroN2022/toy-face'  (a popular SD 1.5 LoRA)
LORA_WEIGHTS = 'path/to/your_lora.safetensors'  # or Hub ID

# TODO: Fill in src/image/lora.py before running.
lora_pipe = LoRAPipeline(
    base_model_id=BASE_MODEL_ID,
    lora_weights=LORA_WEIGHTS,
    lora_scale=0.8,
)
print('LoRA pipeline loaded.')

In [ ]:
# Generate with LoRA
# Replace the prompt's trigger word with the one used in your LoRA training.
LORA_PROMPT = 'sks person hiking in the mountains, golden hour, dramatic lighting'

lora_img = lora_pipe.generate(
    prompt=LORA_PROMPT,
    seed=7,
    num_inference_steps=30,
)
lora_img.save(OUTPUT_DIR / 'lora_output.png')
display(lora_img)

In [ ]:
# LoRA scale sweep — how much the adapter influences the output
for scale in [0.2, 0.5, 0.8, 1.0]:
    lora_pipe_scaled = LoRAPipeline(
        base_model_id=BASE_MODEL_ID,
        lora_weights=LORA_WEIGHTS,
        lora_scale=scale,
    )
    img = lora_pipe_scaled.generate(
        prompt=LORA_PROMPT,
        cross_attention_kwargs={'scale': scale},
        seed=7,
    )
    img.save(OUTPUT_DIR / f'lora_scale_{scale:.1f}.png')
    print(f'lora_scale={scale}')
    display(img)

---
## Part C — DreamBooth

DreamBooth produces a fully fine-tuned model checkpoint.  Train your own using
`src/training/train_dreambooth.py`, then point `DREAMBOOTH_MODEL_ID` at the output.

In [ ]:
from src.image.dreambooth import DreamBoothPipeline

# TODO: Replace with the path or Hub ID of your DreamBooth fine-tuned model.
DREAMBOOTH_MODEL_ID = '../outputs/dreambooth'  # local path after training
SUBJECT_TOKEN = 'sks dog'  # Must match the token used during training

# TODO: Fill in src/image/dreambooth.py before running.
db_pipe = DreamBoothPipeline(
    model_id=DREAMBOOTH_MODEL_ID,
    subject_token=SUBJECT_TOKEN,
)
print('DreamBooth pipeline loaded.')

In [ ]:
# Generate with DreamBooth — the subject token links to the fine-tuned identity
for scene in ['sitting in a park', 'wearing sunglasses on the beach', 'jumping through a puddle']:
    prompt = f'{SUBJECT_TOKEN} {scene}, photorealistic, detailed'
    img = db_pipe.generate(prompt=prompt, seed=0, num_inference_steps=50)
    img.save(OUTPUT_DIR / f'dreambooth_{scene.replace(" ", "_")}.png')
    print(f'Scene: {scene}')
    display(img)

---
## Summary: Technique comparison

| Technique | Training time | VRAM | Fidelity | Flexibility |
|---|---|---|---|---|
| IP-Adapter | None | Low | Medium | High (any image) |
| LoRA | Minutes–hours | Medium | High | High (composable) |
| DreamBooth | Hours | High | Very high | Low (fixed subject) |

→ Continue to **Notebook 03** for consistency scoring and prompt anchor experiments.